# Transformação e agregação de dados por identificador "Ano"

## Configuração do Ambiente

In [0]:
from pyspark.sql.utils import AnalysisException
from src.utils.udfs import functions_for_df_structure_management as ffdsm

## Ingestão de dados da camada de bronze

In [0]:
df_assets_and_rights = spark.table("fiap_1ctor.bronze_layer.delta_bens_e_direitos")

In [0]:
df_debts_and_liabilities = spark.table("fiap_1ctor.bronze_layer.delta_dividas_e_onus")

In [0]:
df_payments_and_donations = spark.table("fiap_1ctor.bronze_layer.delta_pagamentos_e_doacoes")

## Transformação de Dados

### "Bens e Direitos"

In [0]:
df_assets_and_rights = df_assets_and_rights.dropna(how='all')

In [0]:
df_casted_assets_and_rights = ffdsm.cast_columns_to_float(df_assets_and_rights, ["AnoCalendario"])

In [0]:
df_casted_assets_and_rights = ffdsm.rename_columns_with_df_name(df_casted_assets_and_rights, "BensEDireitos", ["AnoCalendario"])

In [0]:
dbutils.data.summarize(df_casted_assets_and_rights)

In [0]:
df_filled_assets_and_rights = ffdsm.fill_nulls(df_casted_assets_and_rights, ["AnoCalendario"])

### "Dívidas e Onus"

In [0]:
df_debts_and_liabilities = df_debts_and_liabilities.dropna(how='all')

In [0]:
df_casted_debts_and_liabilities = ffdsm.cast_columns_to_float(df_debts_and_liabilities, ["AnoCalendario"])

In [0]:
df_casted_debts_and_liabilities = ffdsm.rename_columns_with_df_name(df_casted_debts_and_liabilities, "DividasEOnus", ["AnoCalendario"])

In [0]:
dbutils.data.summarize(df_casted_debts_and_liabilities)

In [0]:
df_filled_debts_and_liabilities = ffdsm.fill_nulls(df_casted_debts_and_liabilities, ["AnoCalendario"])

### "Pagamentos e Doações"

In [0]:
df_payments_and_donations = df_payments_and_donations.dropna(how='all')

In [0]:
df_casted_payments_and_donations = ffdsm.cast_columns_to_float(df_payments_and_donations, ["AnoCalendario"])

In [0]:
df_casted_payments_and_donations = ffdsm.rename_columns_with_df_name(df_casted_payments_and_donations, "PagamentosEDoacoes", ["AnoCalendario"])

In [0]:
dbutils.data.summarize(df_casted_payments_and_donations)

In [0]:
df_filled_debts_and_liabilities = ffdsm.fill_nulls(df_casted_payments_and_donations, ["AnoCalendario"])

## Agregação por "Ano"

In [0]:
df_joined_assets_and_rights_debts_and_liabilities = df_casted_assets_and_rights.join(df_casted_debts_and_liabilities, on="AnoCalendario", how="inner")

In [0]:
df_joined_assets_and_rights_debts_and_liabilities_payments_and_donations = df_joined_assets_and_rights_debts_and_liabilities.join(df_casted_payments_and_donations, on="AnoCalendario", how="inner")
display(df_joined_assets_and_rights_debts_and_liabilities_payments_and_donations)

In [0]:
df_silver_assets_debts_payments = ffdsm.fill_nulls(df_joined_assets_and_rights_debts_and_liabilities_payments_and_donations, ["AnoCalendario"])

## Salvar como Delta na Camada Silver

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS fiap_1ctor.silver_layer")

In [0]:
error = None

try:
    df_silver_assets_debts_payments.write \
        .mode("overwrite") \
        .saveAsTable(f"fiap_1ctor.silver_layer.delta_bens_dividas_pagamentos")
    error = None
except Exception as e:
    error = str(e)
    print(error)